# Pythia 2.8b

In [1]:
import sys
sys.path.insert(0, '../..')
from src.models import load_model, unload

### Load Model

In [2]:
model_name = "pythia-2.8b"
model, info = load_model(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-2.8b into HookedTransformer
Loaded pythia-2.8b on mps
  32 layers | 32 heads | d_model=2560 | d_mlp=10240 | parallel attn+MLP


### Baseline Prompt

In [3]:
prompt = "A screen reader is"
output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
print(f"Input: {prompt}")
print(f"Output: {output}")

# Cache internal states
logits, cache = model.run_with_cache(prompt)
print(f"\nCached {len(cache)} different activation points!")

Input: A screen reader is
Output: A screen reader is a software application that reads aloud the text on a

Cached 579 different activation points!


### Declarative Prompts

In [4]:
test_prompts = [
    "A screen reader is",
    "WCAG stands for", 
    "A skip link is",
    "The purpose of alt text is",
    "ARIA stands for",
    "A focus indicator is",
    "Keyboard navigation allows",
    "Color contrast is important because",
    "Semantic HTML helps",
    "Captions are used for",
]

for prompt in test_prompts:
    output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

A screen reader is             →  a software application that reads aloud the text on a
WCAG stands for                →  the World Council of Churches. The WCC
A skip link is                 →  a link that is used to skip a section of
The purpose of alt text is     →  to provide a brief description of the image.

ARIA stands for                →  “A Rational Approach to Information and Automation
A focus indicator is           →  a device that is used to indicate the focus of
Keyboard navigation allows     →  you to navigate through the pages of this website.
Color contrast is important because →  it is a key factor in determining the quality of
Semantic HTML helps            →  you create semantic HTML documents.

Semantic
Captions are used for          →  the purpose of identifying the subject of a photograph.


### Evaluative Prompts

In [5]:
code_prompts = [
    "The following code is not accessible because it doesn't have what? <img src='photo.jpg'>",
    "A <div> with onclick is not accessible because",
    "The accessibility problem with <a href='#'></a> is",
    "<input type='text'> needs a",
    "A button that only says 'Click here' is bad because",
]
for prompt in code_prompts:
    output = model.generate(prompt, max_new_tokens=20, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

The following code is not accessible because it doesn't have what? <img src='photo.jpg'> → 

The following code is not accessible because it doesn't have what? <img src='photo
A <div> with onclick is not accessible because →  it is not in the same document as the <a> that is clicked.

A <
The accessibility problem with <a href='#'></a> is →  that it is not a valid HTML tag.

The accessibility problem with <a href='#'
<input type='text'> needs a    →  value

<input type='text' value=''>

<input type='text'
A button that only says 'Click here' is bad because →  it's not clear what the button does.

A button that says 'Click here' is


### Perplexity

In [6]:
import torch

correct = "A screen reader is software that reads text aloud for blind users."
wrong = "A screen reader is a device for viewing screens."

def get_perplexity(model, text):
    tokens = model.to_tokens(text)
    logits = model(tokens)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    
    # Get log prob of each actual next token
    token_log_probs = log_probs[0, :-1, :].gather(1, tokens[0, 1:].unsqueeze(1)).squeeze()
    
    # Perplexity = exp(-mean(log_probs))
    return torch.exp(-token_log_probs.mean()).item()

print(f"Correct: {get_perplexity(model, correct)}")
print(f"Wrong: {get_perplexity(model, wrong)}")

Correct: 13.648499488830566
Wrong: 54.90721893310547


### Attention Binding

In [7]:
import pandas as pd

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices

logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

out_path = (f"../results/{model_name}")
df = pd.DataFrame(rows)
df.to_csv(f"../results/{model_name}/attention-binding.csv", index=False)


print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")    

print(f"\nFound {len(rows)} heads above threshold")
print(f"Saved to {out_path}")

[(0, '<|endoftext|>'), (1, 'A'), (2, ' screen'), (3, ' reader'), (4, ' is')]

Top 10 by binding strength:
Layer  1, Head 12: 0.9909
Layer  1, Head 29: 0.984
Layer  1, Head  6: 0.9815
Layer  0, Head 21: 0.9726
Layer  1, Head 17: 0.9719
Layer  1, Head 25: 0.9483
Layer  1, Head 11: 0.9268
Layer  1, Head 22: 0.9058
Layer 29, Head  7: 0.9019
Layer  4, Head 16: 0.8896

Found 101 heads above threshold
Saved to ../results/pythia-2.8b


In [8]:
import circuitsvis as cv

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 11
attention = cache["pattern", layer]  # shape: [heads, seq, seq]
print(f"Layer 1, Head 12")

html = cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

html

Layer 1, Head 12


In [26]:
unload(model)

Model unloaded, memory cleared
